In [1]:
from copy import deepcopy
import torch
import sys
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda import amp
from spikingjelly.activation_based import functional, surrogate, neuron, layer
from spikingjelly.activation_based.model import parametric_lif_net
from spikingjelly.datasets.dvs128_gesture import DVS128Gesture
from torch.utils.data import DataLoader
import time
import os
import argparse
import datetime

In [2]:
torch.manual_seed(1)

In [3]:
T = 4
b = 8
j = 8
lr = 0.001
epochs = 30
channels = 128

data_dir = os.path.expanduser('~/datasets/DVSGesture/')

In [4]:
device = 'cuda:0'

In [5]:
class DVSGestureNet(nn.Module):
    def __init__(self, channels=32, spiking_neuron: callable=None, is_seperable=False, kernel_size=3, **kwargs):
        super().__init__()

        conv = []
        ## Stem
        conv.append(layer.Conv2d(2, channels, kernel_size=2, stride=2, 
                                    bias=False))
        conv.append(layer.BatchNorm2d(channels))

        ## Middle Layers
        for i in range(4):
            if is_seperable:
                conv.append(layer.Conv2d(channels, channels,
                                         kernel_size=kernel_size, groups=channels,
                                         padding='same', bias=False)
                )
                conv.append(layer.Conv2d(channels, channels, kernel_size=1))
            else:
                conv.append(layer.Conv2d(channels, channels, kernel_size=kernel_size, 
                                         padding='same', bias=False))
            conv.append(layer.BatchNorm2d(channels))    
            conv.append(spiking_neuron(**deepcopy(kwargs)))
            if i != 3:
                conv.append(layer.Conv2d(channels, 2*channels, kernel_size=2, bias=False, stride=2))
                channels = channels * 2
        

        self.conv = nn.Sequential(
            *conv, 
            layer.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = layer.Linear(in_features=channels, out_features=11)
        
    def forward(self, x: torch.Tensor):
        x = self.conv(x).mean((3, 4))
        x = self.fc(x)
            
        return x


In [6]:
train_set = DVS128Gesture(root=data_dir, train=True, data_type='frame', frames_number=T, split_by='number')
test_set = DVS128Gesture(root=data_dir, train=False, data_type='frame', frames_number=T, split_by='number')

Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number].
Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number/train].
Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number/train/4].
Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number/train/6].
Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number/train/1].
Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number/train/2].
Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number/train/0].
Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number/train/10].
Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number/train/3].
Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number/train/9].
Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number/train/5].
Mkdir [/home/tahaf/datasets/DVSGesture/frames_number_4_split_by_number/train/8].
Mkdir [/home/tahaf/datasets/DVSGestur

In [11]:
train_data_loader = torch.utils.data.DataLoader(
    dataset=train_set,
    batch_size=b,
    shuffle=True,
    drop_last=True,
    num_workers=j,
    pin_memory=True
)

test_data_loader = torch.utils.data.DataLoader(
    dataset=test_set,
    batch_size=b,
    shuffle=True,
    drop_last=False,
    num_workers=j,
    pin_memory=True
)

In [12]:
scaler = amp.GradScaler()

In [13]:
def check_model(kernel_size, channels, is_seperable):
    print(f'Results for kernel size = {kernel_size} and seperable convolution = {is_seperable} and channels = {channels}')
    max_test_acc = -1

    net = DVSGestureNet(
        channels=channels,
        kernel_size=kernel_size,
        spiking_neuron=neuron.LIFNode,
        surrogate_function=surrogate.ATan(),
        is_seperable=is_seperable,
        detach_reset=True
    )
    net.to(device)
    num_params = sum(p.numel() for p in net.parameters())
    functional.set_step_mode(net, step_mode='m')

    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, total_steps=round(epochs*60000/b), max_lr=lr)
    
    start_time = time.time()
    for epoch in range(epochs):
        net.train()
        train_loss = 0
        train_acc = 0
        train_samples = 0
        for frame, label in train_data_loader:
            optimizer.zero_grad()
            frame = frame.to(device)
            frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
            label = label.to(device)
            label_onehot = F.one_hot(label, 11).float()
    
            if scaler is not None:
                with amp.autocast():
                    out_fr = net(frame)
                    loss = functional.temporal_efficient_training_cross_entropy(out_fr, label)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out_fr = net(frame)
                loss = functional.temporal_efficient_training_cross_entropy(out_fr, label)
                loss.backward()
                optimizer.step()
    
            train_samples += label.numel()
            train_loss += loss.item() * label.numel()
            train_acc += (out_fr.mean(0).argmax(1) == label).float().sum().item()
    
            functional.reset_net(net)
    
        train_loss /= train_samples
        train_acc /= train_samples
    
        lr_scheduler.step()
    
        net.eval()
        test_loss = 0
        test_acc = 0
        test_samples = 0
        with torch.no_grad():
            for frame, label in test_data_loader:
                frame = frame.to(device)
                frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
                label = label.to(device)
                label_onehot = F.one_hot(label, 11).float()
                out_fr = net(frame)
                loss = functional.temporal_efficient_training_cross_entropy(out_fr, label)
                test_samples += label.numel()
                test_loss += loss.item() * label.numel()
                test_acc += (out_fr.mean(0).argmax(1) == label).float().sum().item()
                functional.reset_net(net)
        test_loss /= test_samples
        test_acc /= test_samples
        max_test_acc = max(max_test_acc, test_acc)
        
        print(f'epoch = {epoch}, train_loss ={train_loss: .4f}, train_acc ={train_acc: .4f}, test_loss ={test_loss: .4f}, test_acc ={test_acc: .4f}, max_test_acc ={max_test_acc: .4f}')
    end_time = time.time()
    total_time = end_time - start_time
    print(f'total time: {total_time}s')
    print('-'*1000)

    return {
        'accuracy': max_test_acc,
        'total_time': total_time,
        'num_params': num_params,
    }

In [14]:
kernel_sizes = [3, 5, 7]
channels = [32, 64]
results = {}

In [ ]:
for channel in channels:
    results[channel] = {}
    for kernel_size in kernel_sizes:
        results[channel][kernel_size] = {}
        for is_seperable in [False, True]:
            result = check_model(channels=channel, kernel_size=kernel_size, is_seperable=is_seperable)
            results[channel][kernel_size][is_seperable] = result

Results for kernel size = 3 and seperable convolution = False and channels = 32
epoch = 0, train_loss = 2.3436, train_acc = 0.1862, test_loss = 2.2546, test_acc = 0.3646, max_test_acc = 0.3646
epoch = 1, train_loss = 2.2255, train_acc = 0.3087, test_loss = 2.1208, test_acc = 0.4201, max_test_acc = 0.4201
epoch = 2, train_loss = 2.1136, train_acc = 0.3588, test_loss = 2.0333, test_acc = 0.4549, max_test_acc = 0.4549
epoch = 3, train_loss = 2.0281, train_acc = 0.4056, test_loss = 1.9304, test_acc = 0.4688, max_test_acc = 0.4688
epoch = 4, train_loss = 1.9489, train_acc = 0.4201, test_loss = 1.8145, test_acc = 0.4688, max_test_acc = 0.4688
epoch = 5, train_loss = 1.8802, train_acc = 0.4507, test_loss = 1.8128, test_acc = 0.4861, max_test_acc = 0.4861
epoch = 6, train_loss = 1.8235, train_acc = 0.4575, test_loss = 1.7717, test_acc = 0.4896, max_test_acc = 0.4896
epoch = 7, train_loss = 1.7702, train_acc = 0.4702, test_loss = 1.6802, test_acc = 0.5104, max_test_acc = 0.5104
epoch = 8, train

In [ ]:
results

In [ ]:
import os
import json

with open(os.path.expanduser('~/new_results.json'), 'w') as f:
    json.dump(results, f, indent=6) 